# Beats and group velocity

*waves · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/beats_superposition)

Add up two (or more) sinusoids with close frequencies. All waves sit in one tensor indexed by wave $s$, time $n$ and position $j$; a single einsum over $s$ superposes them. At a fixed point you hear beats at $|f_1 - f_2|$; in space the carrier crests move at the phase velocity while the envelope moves at the group velocity.

**Vector form**

$$y(x, t) = \sum_s A_s\cos(k_s x - \omega_s t),\qquad v_p = \frac{\bar\omega}{\bar k},\qquad v_g = \frac{\Delta\omega}{\Delta k}$$

**Index (tensor) form**

$$y_{nj} = A_s\cos\theta_{snj},\qquad \theta_{snj} = k_s\,x_j - \omega_s\,t_n,\qquad \mathcal{E}_{nj} = \bigl|A_s\,e^{i\theta_{snj}}\bigr|$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\omega_s,\ k_s$ | `omega, k` | (S,) | angular frequency and wavenumber of wave $s$, linked by the dispersion relation $\omega \propto k^p$ |
| $x_j,\ t_n$ | `x, t` | (J,), (N,) | positions and times where the wave is evaluated |
| $\theta_{snj}$ | `phase(k, omega, x, t)` | (S, N, J) | phase of every wave at every time and place, by broadcasting |
| $A_s\cos\theta_{snj}$ | `einsum('s,snj->nj', A, cos(theta))` | (N, J) | repeated $s$ ⇒ summed: the superposition |
| $\mathcal{E}_{nj}$ | `einsum('s,snj->nj', A, exp(1j*theta)).abs()` | (N, J) | envelope: length of the sum of phasors |

For two waves $\cos a + \cos b = 2\cos\tfrac{a-b}{2}\cos\tfrac{a+b}{2}$: a fast carrier at the mean frequency inside a slow envelope. The loudness $|{\rm envelope}|$ peaks $|f_1 - f_2|$ times per second, the beat frequency. With the dispersion relation $\omega = c_0 k_0 (k/k_0)^p$ the group velocity is $p$ times the phase velocity: equal for sound on a string ($p = 1$), half for deep-water waves ($p = \tfrac12$), one and a half for capillary ripples ($p = \tfrac32$). The red dot rides a crest at $v_p$, the green dot rides the envelope peak at $v_g$. With more than two waves the beats become isolated wave groups.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/beats_superposition/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    f1, f2 = sorted((params["f1"], params["f2"]))
    if f2 - f1 < 1e-6:
        f2 = f1 + 0.05  # identical frequencies would give no beats at all
    omega = tensors.angular_frequencies(f1, f2, params["n_waves"], dev)
    k = tensors.wavenumbers(omega, params["dispersion"])
    A = tensors.amplitudes(params["n_waves"], dev)

    v_p, v_g = equation.phase_velocity(k, omega), equation.group_velocity(k, omega)
    x_max = max(10.0, v_p * params["t_max"])  # wide enough to follow the crest rider
    x, t = tensors.grids(x_max, params["t_max"], dev)
    result = solve.solve(A, k, omega, x, t)

    dt = (t[1] - t[0]).item()
    print(f"phase velocity v_p = {v_p:.3f} m/s,  group velocity v_g = {v_g:.3f} m/s,  ratio {v_g / v_p:.3f}")
    f_beat = solve.beat_frequency(result["trace"], dt, f1)
    print(f"beat frequency measured {f_beat:.3f} Hz   (|f1 - f2| = {f2 - f1:.3f} Hz)")
    return plot.animate(x, t, result, v_p, v_g, out_dir / "beats_superposition.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/beats_superposition/equation.py
r"""Physics → tensor translation.

Superposition of S travelling sinusoids,

    y(x, t) = \sum_s A_s \cos(k_s x - \omega_s t),

in index form with positions x_j and times t_n:

    \theta_{snj} = k_s x_j - \omega_s t_n,        y_{nj} = A_s \cos\theta_{snj}     (s is summed)

The envelope is the length of the sum of phasors, \mathcal E_{nj} = | A_s e^{i\theta_{snj}} |  (s summed).
For two waves the carrier moves at v_p = \bar\omega / \bar k and the envelope at v_g = \Delta\omega / \Delta k.
"""

import torch


def phase(k: torch.Tensor, omega: torch.Tensor, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    r"""θ_{snj} = k_s x_j − ω_s t_n by broadcasting.   k, omega: (S,), x: (J,), t: (N,) → (S, N, J)"""
    return k[:, None, None] * x[None, None, :] - omega[:, None, None] * t[None, :, None]


def superpose(A: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    r"""y_{nj} = A_s cos θ_{snj}   — s summed.   A: (S,), theta: (S, N, J) → (N, J)"""
    return torch.einsum("s,snj->nj", A, torch.cos(theta))


def envelope(A: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    r"""\mathcal E_{nj} = | A_s e^{iθ_{snj}} |   — s summed.   → (N, J)"""
    phasors = torch.einsum("s,snj->nj", A.to(torch.complex64), torch.exp(1j * theta))
    return phasors.abs()


def phase_velocity(k: torch.Tensor, omega: torch.Tensor) -> float:
    """v_p = mean ω / mean k (the carrier)."""
    return (omega.mean() / k.mean()).item()


def group_velocity(k: torch.Tensor, omega: torch.Tensor) -> float:
    """v_g = Δω / Δk between the outermost waves (the envelope)."""
    return ((omega[-1] - omega[0]) / (k[-1] - k[0])).item()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/beats_superposition/tensors.py
"""Build the concrete tensors: frequencies, wavenumbers from the dispersion relation, amplitudes and grids."""

import math

import torch

C0 = 1.0  # phase velocity (m/s) at the reference wavenumber
K0 = 2 * math.pi  # reference wavenumber (rad/m): a 1 Hz wave is 1 m long
POWERS = {"none": 1.0, "water": 0.5, "capillary": 1.5}  # ω ∝ k^p


def angular_frequencies(f1: float, f2: float, n_waves: int, device: torch.device) -> torch.Tensor:
    """ω_s = 2π f_s with f_s evenly spread from f1 to f2: (S,)."""
    return 2 * math.pi * torch.linspace(f1, f2, n_waves, device=device)


def wavenumbers(omega: torch.Tensor, dispersion: str) -> torch.Tensor:
    """Invert ω = c0 k0 (k / k0)^p  →  k = k0 (ω / (c0 k0))^{1/p}: (S,)."""
    p = POWERS[dispersion]
    return K0 * (omega / (C0 * K0)) ** (1.0 / p)


def amplitudes(n_waves: int, device: torch.device) -> torch.Tensor:
    """A_s = 1/S so the sum never exceeds 1: (S,)."""
    return torch.full((n_waves,), 1.0 / n_waves, device=device)


def grids(x_max: float, t_max: float, device: torch.device, nx: int = 400, dt: float = 0.02):
    """x (J,) for the snapshot and t (N,) for the time trace."""
    x = torch.linspace(0.0, x_max, nx, device=device)
    t = torch.arange(0.0, t_max + 1e-9, dt, device=device)
    return x, t


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/beats_superposition/solve.py
"""Evaluate the superposition everywhere at once (no time stepping needed: every wave is known exactly)."""

import torch

from .equation import envelope, phase, superpose


def solve(A: torch.Tensor, k: torch.Tensor, omega: torch.Tensor, x: torch.Tensor, t: torch.Tensor) -> dict:
    """Return the field y (N, J), its envelope (N, J), and the time trace at x = 0 with its envelope (N,)."""
    theta = phase(k, omega, x, t)  # (S, N, J)
    y = superpose(A, theta)
    env = envelope(A, theta)
    return {"y": y, "env": env, "trace": y[:, 0], "trace_env": env[:, 0]}


def beat_frequency(trace: torch.Tensor, dt: float, f_min: float) -> float:
    """Measure the beat frequency from the time trace at one point.

    The loudness y² contains slow terms cos((ω1 − ω2) t) and fast terms near 2 f̄. Below 2 f_min only the
    slow ones are left, and the strongest of them is the beat frequency |f1 − f2|.
    """
    power = trace**2 - (trace**2).mean()
    spectrum = torch.fft.rfft(power).abs()  # (N/2 + 1,)
    freqs = torch.fft.rfftfreq(len(trace), d=dt).to(trace.device)
    slow = (freqs > 0) & (freqs < 1.9 * f_min)
    return freqs[slow][spectrum[slow].argmax()].item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/beats_superposition/plot.py
"""Top: the wave in space with its envelope and two riders (crest at v_p, envelope peak at v_g).
Bottom: the time trace at x = 0, where the beats are heard."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(x: torch.Tensor, t: torch.Tensor, result: dict, v_p: float, v_g: float, path: Path) -> Path:
    xs, ts = x.cpu().numpy(), t.cpu().numpy()
    y, env = result["y"].cpu().numpy(), result["env"].cpu().numpy()  # (N, J)
    trace, trace_env = result["trace"].cpu().numpy(), result["trace_env"].cpu().numpy()  # (N,)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.3))
    line = ax1.plot(xs, y[0], lw=1.5, color="#2563eb")[0]
    env_hi = ax1.plot(xs, env[0], "--", color="#9ca3af", lw=1)[0]
    env_lo = ax1.plot(xs, -env[0], "--", color="#9ca3af", lw=1)[0]
    crest = ax1.plot([], [], "o", color="#dc2626", ms=7, label=f"crest  v_p = {v_p:.2f} m/s")[0]
    group = ax1.plot([], [], "o", color="#16a34a", ms=7, label=f"envelope  v_g = {v_g:.2f} m/s")[0]
    ax1.set_xlim(xs[0], xs[-1])
    ax1.set_ylim(-1.15, 1.5)
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y")
    ax1.legend(loc="upper right", fontsize=8, ncol=2)
    title = ax1.set_title("snapshot y(x, t)")

    trace_line = ax2.plot([], [], lw=1, color="#2563eb")[0]
    trace_env_line = ax2.plot([], [], "--", color="#9ca3af", lw=1)[0]
    ax2.set_xlim(ts[0], ts[-1])
    ax2.set_ylim(-1.15, 1.15)
    ax2.set_xlabel("t (s)")
    ax2.set_ylabel("y(0, t)")

    def update(step: int):
        line.set_ydata(y[step])
        env_hi.set_ydata(env[step])
        env_lo.set_ydata(-env[step])
        tn = ts[step]
        # riders start at x = 0 (a crest and the envelope peak coincide there at t = 0)
        x_c, x_g = v_p * tn, v_g * tn
        for dot, xr in ((crest, x_c), (group, x_g)):
            visible = xr <= xs[-1]
            dot.set_data([xr] if visible else [], [np.interp(xr, xs, env[step])] if visible else [])
        trace_line.set_data(ts[: step + 1], trace[: step + 1])
        trace_env_line.set_data(ts[: step + 1], trace_env[: step + 1])
        title.set_text(f"snapshot y(x, t)    t = {tn:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(ts), 80), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'f1': 1.0,
    'f2': 1.1,
    'n_waves': 2,
    'dispersion': 'water',
    't_max': 20.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.beats_superposition")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.beats_superposition.main")
problem_dir = Path("problems/waves/beats_superposition")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
